# ShellWhisperer-1.5B Fine-Tuning with Unsloth

Fine-tune `Qwen/Qwen2.5-Coder-1.5B-Instruct` on shell/command-line data.

**Runtime**: GPU T4 (free Colab) | **Time**: ~15 min | **VRAM**: ~6GB


In [ ]:
%%capture
!pip install unsloth
!pip uninstall -y transformers && pip install transformers==4.46.3

from unsloth import FastLanguageModel
import torch

In [ ]:
# Configuration
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
HF_REPO = "fableforge-ai/ShellWhisperer-1.5B"
MAX_SEQ_LENGTH = 2048
LORA_R = 16
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1

# Load model with LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,  # Auto-detect
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# Prepare training data
from datasets import Dataset
import json

# Upload shellwhisperer_train.jsonl and shellwhisperer_val.jsonl
# to this Colab's session storage, or load from HuggingFace
TRAIN_PATH = "shellwhisperer_train.jsonl"
VAL_PATH = "shellwhisperer_val.jsonl"

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

raw_data = load_jsonl(TRAIN_PATH)
val_data = load_jsonl(VAL_PATH)

# Convert Alpaca format to ChatML
def to_chatml(example):
    if "messages" in example:
        return example  # Already ChatML
    return {
        "messages": [
            {"role": "system", "content": "You are ShellWhisperer, an expert in command-line tools, shell scripting, and system administration. Provide concise, accurate commands with brief explanations."},
            {"role": "user", "content": example["instruction"] + ("\n" + example["input"] if example.get("input") else "")},
            {"role": "assistant", "content": example["output"]}
        ]
    }

train_data = [to_chatml(ex) for ex in raw_data]
val_converted = [to_chatml(ex) for ex in val_data]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_converted)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
# Format for training
def format_to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_to_text)
val_dataset = val_dataset.map(format_to_text)

print(train_dataset[0]["text"][:300])

In [ ]:
# Train!
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.1f}s")

In [ ]:
# Test inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are ShellWhisperer, an expert in command-line tools, shell scripting, and system administration."},
    {"role": "user", "content": "Find all Python files modified in the last week"},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.3,
    top_p=0.9,
    use_cache=True,
)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# Save LoRA adapters locally
model.save_pretrained("shellwhisperer-lora")
tokenizer.save_pretrained("shellwhisperer-lora")
print("LoRA adapters saved to ./shellwhisperer-lora")

In [ ]:
# Merge LoRA into base model and save full weights
model.save_pretrained_merged("shellwhisperer-merged", tokenizer)
print("Merged model saved to ./shellwhisperer-merged")

In [ ]:
# Push to HuggingFace (requires login)
# Run: huggingface-cli login
# Or set HF_TOKEN in Colab secrets

# Push LoRA adapters
# model.push_to_hub(HF_REPO + "-LoRA", private=False)
# tokenizer.push_to_hub(HF_REPO + "-LoRA", private=False)

# Push merged model
# model.push_to_hub_merged(HF_REPO, tokenizer)
# print(f"Model pushed to https://huggingface.co/{HF_REPO}")